In [3]:
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# =========================
# 1. 파일 로드
# =========================
train_path = "klue_ner_train.json"
valid_path = "klue_ner_valid.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(valid_path, "r", encoding="utf-8") as f:
    valid_data = json.load(f)

print("train size:", len(train_data))
print("valid size:", len(valid_data))
print(train_data[0].keys())
print(train_data[0])

train size: 21008
valid size: 5000
dict_keys(['id', 'sentence', 'tokens', 'tags', 'tag_ids'])
{'id': 0, 'sentence': '특히 <영동고속도로:LC> <강릉:LC> 방향 <문막휴게소:LC>에서 <만종분기점:LC>까지 <5㎞:QT> 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.', 'tokens': ['특', '히', ' ', '영', '동', '고', '속', '도', '로', ' ', '강', '릉', ' ', '방', '향', ' ', '문', '막', '휴', '게', '소', '에', '서', ' ', '만', '종', '분', '기', '점', '까', '지', ' ', '5', '㎞', ' ', '구', '간', '에', '는', ' ', '승', '용', '차', ' ', '전', '용', ' ', '임', '시', ' ', '갓', '길', '차', '로', '제', '를', ' ', '운', '영', '하', '기', '로', ' ', '했', '다', '.'], 'tags': ['O', 'O', 'O', 'B-LC', 'I-LC', 'I-LC', 'I-LC', 'I-LC', 'I-LC', 'O', 'B-LC', 'I-LC', 'O', 'O', 'O', 'O', 'B-LC', 'I-LC', 'I-LC', 'I-LC', 'I-LC', 'O', 'O', 'O', 'B-LC', 'I-LC', 'I-LC', 'I-LC', 'I-LC', 'O', 'O', 'O', 'B-QT', 'I-QT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'tag_ids': [12, 12, 12, 2, 3, 3, 3, 3, 3, 12, 2

In [4]:
label_names = [
    "B-DT", "I-DT",
    "B-LC", "I-LC",
    "B-OG", "I-OG",
    "B-PS", "I-PS",
    "B-QT", "I-QT",
    "B-TI", "I-TI",
    "O"
]

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

print(label2id)

{'B-DT': 0, 'I-DT': 1, 'B-LC': 2, 'I-LC': 3, 'B-OG': 4, 'I-OG': 5, 'B-PS': 6, 'I-PS': 7, 'B-QT': 8, 'I-QT': 9, 'B-TI': 10, 'I-TI': 11, 'O': 12}


In [9]:
from transformers import AutoTokenizer

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
def tokenize_and_align_labels_from_json(example, max_length=128):
    """
    example:
    {
        "sentence": ...,
        "tokens": [...],
        "tags": [...],
        "tag_ids": [...]
    }
    """

    tokens = example["tokens"]
    tag_ids = example["tag_ids"]

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_attention_mask=True
    )

    word_ids = encoding.word_ids()
    aligned_labels = []

    prev_word_idx = None
    o_id = label2id["O"]

    for word_idx in word_ids:
        # [CLS], [SEP], [PAD]
        if word_idx is None:
            aligned_labels.append(o_id)

        # 어떤 원래 토큰의 첫 번째 subword
        elif word_idx != prev_word_idx:
            aligned_labels.append(tag_ids[word_idx])

        # 같은 원래 토큰의 뒤 subword
        else:
            curr_label_id = tag_ids[word_idx]
            curr_label_name = id2label[curr_label_id]

            if curr_label_name.startswith("B-"):
                inside_label = "I-" + curr_label_name[2:]
                aligned_labels.append(label2id.get(inside_label, curr_label_id))
            else:
                aligned_labels.append(curr_label_id)

        prev_word_idx = word_idx

    item = {
        "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
        "labels": torch.tensor(aligned_labels, dtype=torch.long),
    }

    return item

In [11]:
class JsonNERDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        return tokenize_and_align_labels_from_json(example, max_length=self.max_length)

In [12]:
max_length = 128
batch_size = 32

train_dataset = JsonNERDataset(train_data, tokenizer, max_length=max_length)
valid_dataset = JsonNERDataset(valid_data, tokenizer, max_length=max_length)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

batch = next(iter(train_loader))
print(batch["input_ids"].shape)      # [batch, seq_len]
print(batch["attention_mask"].shape) # [batch, seq_len]
print(batch["labels"].shape)         # [batch, seq_len]

torch.Size([32, 128])
torch.Size([32, 128])
torch.Size([32, 128])


In [15]:
sample = train_dataset[0]

tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"].tolist())
labels = sample["labels"].tolist()
mask = sample["attention_mask"].tolist()

for i, (tok, lab, m) in enumerate(zip(tokens, labels, mask)):
    print(f"{i:3d} | {tok:12} | mask={m} | {lab:2d} | {id2label[lab]}")

  0 | [CLS]        | mask=1 | 12 | O
  1 | 특            | mask=1 | 12 | O
  2 | 히            | mask=1 | 12 | O
  3 | 영            | mask=1 |  2 | B-LC
  4 | 동            | mask=1 |  3 | I-LC
  5 | 고            | mask=1 |  3 | I-LC
  6 | 속            | mask=1 |  3 | I-LC
  7 | 도            | mask=1 |  3 | I-LC
  8 | 로            | mask=1 |  3 | I-LC
  9 | 강            | mask=1 |  2 | B-LC
 10 | 릉            | mask=1 |  3 | I-LC
 11 | 방            | mask=1 | 12 | O
 12 | 향            | mask=1 | 12 | O
 13 | 문            | mask=1 |  2 | B-LC
 14 | 막            | mask=1 |  3 | I-LC
 15 | 휴            | mask=1 |  3 | I-LC
 16 | 게            | mask=1 |  3 | I-LC
 17 | 소            | mask=1 |  3 | I-LC
 18 | 에            | mask=1 | 12 | O
 19 | 서            | mask=1 | 12 | O
 20 | 만            | mask=1 |  2 | B-LC
 21 | 종            | mask=1 |  3 | I-LC
 22 | 분            | mask=1 |  3 | I-LC
 23 | 기            | mask=1 |  3 | I-LC
 24 | 점            | mask=1 |  3 | I-LC
 25 | 까            | 

In [16]:
!pip install pytorch-crf

In [17]:
import torch
import torch.nn as nn
from transformers import BertModel
from torchcrf import CRF

class BertCRF(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super(BertCRF, self).__init__()

        self.bert = BertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.position_wise_ff = nn.Linear(hidden_size, num_classes)
        self.crf = CRF(num_tags=num_classes, batch_first=True)

    def forward(self, input_ids, attention_mask=None, tags=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.position_wise_ff(sequence_output)

        if attention_mask is not None:
            crf_mask = attention_mask.bool()
        else:
            crf_mask = torch.ones_like(input_ids, dtype=torch.bool)

        if tags is not None:
            log_likelihood = self.crf(
                emissions,
                tags,
                mask=crf_mask,
                reduction='mean'
            )
            loss = -log_likelihood
            predictions = self.crf.decode(emissions, mask=crf_mask)
            return loss, predictions

        else:
            predictions = self.crf.decode(emissions, mask=crf_mask)
            return predictions

In [18]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=bcfd18c6eebb9b5499d704651cf97ce9aebe5cd35b8eb74f7ac8f774067b18cf
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [25]:
import torch
from torch.optim import AdamW
from tqdm import tqdm
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def convert_predictions_to_labels(predictions, labels, attention_mask, input_ids, tokenizer, id2label):
    true_label_list = []
    pred_label_list = []

    if torch.is_tensor(labels):
        labels = labels.cpu().tolist()
    if torch.is_tensor(attention_mask):
        attention_mask = attention_mask.cpu().tolist()
    if torch.is_tensor(input_ids):
        input_ids = input_ids.cpu().tolist()

    for pred_seq, true_seq, mask_seq, input_id_seq in zip(predictions, labels, attention_mask, input_ids):
        true_labels = []
        pred_labels = []

        valid_len = sum(mask_seq)  # CRF decode 길이와 동일
        tokens = tokenizer.convert_ids_to_tokens(input_id_seq[:valid_len])

        for j in range(valid_len):
            tok = tokens[j]

            # special token 제외
            if tok in tokenizer.all_special_tokens:
                continue

            true_labels.append(id2label[true_seq[j]])
            pred_labels.append(id2label[pred_seq[j]])

        true_label_list.append(true_labels)
        pred_label_list.append(pred_labels)

    return true_label_list, pred_label_list


In [26]:
def token_level_accuracy(predictions, labels, attention_mask, input_ids, tokenizer):
    correct = 0
    total = 0

    if torch.is_tensor(labels):
        labels = labels.cpu().tolist()
    if torch.is_tensor(attention_mask):
        attention_mask = attention_mask.cpu().tolist()
    if torch.is_tensor(input_ids):
        input_ids = input_ids.cpu().tolist()

    for pred_seq, true_seq, mask_seq, input_id_seq in zip(predictions, labels, attention_mask, input_ids):
        valid_len = sum(mask_seq)
        tokens = tokenizer.convert_ids_to_tokens(input_id_seq[:valid_len])

        for j in range(valid_len):
            tok = tokens[j]

            # [CLS], [SEP] 제외
            if tok in tokenizer.all_special_tokens:
                continue

            if pred_seq[j] == true_seq[j]:
                correct += 1
            total += 1

    return correct / total if total > 0 else 0.0

In [21]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        loss, _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            tags=labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(batch_loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [27]:
from seqeval.metrics import precision_score, recall_score, f1_score

def evaluate(model, dataloader, device, id2label, tokenizer):
    model.eval()

    total_loss = 0
    all_true_labels = []
    all_pred_labels = []
    total_correct = 0
    total_count = 0

    progress_bar = tqdm(dataloader, desc="Eval", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss, predictions = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                tags=labels
            )

            total_loss += loss.item()

            labels_cpu = labels.cpu().tolist()
            mask_cpu = attention_mask.cpu().tolist()
            input_ids_cpu = input_ids.cpu().tolist()

            for pred_seq, true_seq, mask_seq, input_id_seq in zip(
                predictions, labels_cpu, mask_cpu, input_ids_cpu
            ):
                valid_len = sum(mask_seq)
                tokens = tokenizer.convert_ids_to_tokens(input_id_seq[:valid_len])

                sent_true = []
                sent_pred = []

                for j in range(valid_len):
                    tok = tokens[j]

                    # [CLS], [SEP] 제외
                    if tok in tokenizer.all_special_tokens:
                        continue

                    true_id = true_seq[j]
                    pred_id = pred_seq[j]

                    sent_true.append(id2label[true_id])
                    sent_pred.append(id2label[pred_id])

                    if true_id == pred_id:
                        total_correct += 1
                    total_count += 1

                all_true_labels.append(sent_true)
                all_pred_labels.append(sent_pred)

    avg_loss = total_loss / len(dataloader)
    accuracy = total_correct / total_count if total_count > 0 else 0.0
    precision = precision_score(all_true_labels, all_pred_labels)
    recall = recall_score(all_true_labels, all_pred_labels)
    f1 = f1_score(all_true_labels, all_pred_labels)

    return avg_loss, accuracy, precision, recall, f1, all_true_labels, all_pred_labels

In [28]:
def train_model(model, train_loader, valid_loader, optimizer, device, id2label, tokenizer, epochs=5):
    best_f1 = 0.0

    for epoch in range(epochs):
        print(f"\n===== Epoch {epoch+1}/{epochs} =====")

        train_loss = train_one_epoch(model, train_loader, optimizer, device)

        val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = evaluate(
            model, valid_loader, device, id2label, tokenizer
        )

        print(f"Train Loss     : {train_loss:.4f}")
        print(f"Valid Loss     : {val_loss:.4f}")
        print(f"Valid Accuracy : {val_acc:.4f}")
        print(f"Valid Precision: {val_precision:.4f}")
        print(f"Valid Recall   : {val_recall:.4f}")
        print(f"Valid F1       : {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_bert_crf.pt")
            print("Best model saved.")

    print(f"\nBest Valid F1: {best_f1:.4f}")

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertCRF(
    model_name=model_name,
    num_classes=len(label_names),
    dropout=0.3
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    device=device,
    id2label=id2label,
    tokenizer=tokenizer,
    epochs=5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== Epoch 1/5 =====


Train Loss     : 12.9696
Valid Loss     : 8.3978
Valid Accuracy : 0.9391
Valid Precision: 0.6559
Valid Recall   : 0.7365
Valid F1       : 0.6939
Best model saved.

===== Epoch 2/5 =====


Train Loss     : 6.0109
Valid Loss     : 7.4653
Valid Accuracy : 0.9490
Valid Precision: 0.7327
Valid Recall   : 0.7568
Valid F1       : 0.7445
Best model saved.

===== Epoch 3/5 =====


Train Loss     : 4.2341
Valid Loss     : 6.7592
Valid Accuracy : 0.9551
Valid Precision: 0.7530
Valid Recall   : 0.7858
Valid F1       : 0.7691
Best model saved.

===== Epoch 4/5 =====


Train Loss     : 3.1694
Valid Loss     : 6.7296
Valid Accuracy : 0.9558
Valid Precision: 0.7564
Valid Recall   : 0.7936
Valid F1       : 0.7745
Best model saved.

===== Epoch 5/5 =====


Train Loss     : 2.3870
Valid Loss     : 7.5457
Valid Accuracy : 0.9554
Valid Precision: 0.7624
Valid Recall   : 0.8049
Valid F1       : 0.7831
Best model saved.

Best Valid F1: 0.7831
